# CF Splitting

The C/F (coarse/fine) splitting is the first step in building a multigrid hierarchy. It partitions the unknowns into:
- **F-points** (fine): eliminated on this level by computing an approximation to $\tilde{A}_{ff}^{-1}$
- **C-points** (coarse): representing the coarse grid

PFLARE's **PMISR-DDC** algorithm produces an $A_{ff}$ block that is **diagonally dominant** — the key property enabling high-quality F-point smoothing.

The **PMISR-DDC** does two passes:
1. One pass of **PMISR** which excludes any large entries from the off-diagonal of $A_{ff}$
2. One pass of **DDC** which checks the diagonal dominance ratio of each row in $A_{ff}$ and swaps the rows with largest ratio into C points. This prevents many small off-diagonal entries in $A_{ff}$ from adding up and destroying the diagonal dominance.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare

print("Ready.")

## 1. Build a 1D upwind advection matrix

The 1D steady advection equation $u_x = 0$ with upwind differencing gives a bidiagonal matrix.  This is the canonical asymmetric test case and is lower triangular. 

In [ ]:
def build_1d_upwind(N):
    """Upwind FD matrix for u_x = 0 on [0,1] with Dirichlet left BC."""
    A = PETSc.Mat().create()
    A.setSizes([N, N])
    A.setFromOptions()
    A.setPreallocationNNZ(2)
    A.setUp()

    rstart, rend = A.getOwnershipRange()
    for i in range(rstart, rend):
        A.setValue(i, i, 1.0)         # diagonal
        if i > 0:
            A.setValue(i, i - 1, -1.0)  # upwind lower neighbour
    A.assemblyBegin()
    A.assemblyEnd()
    return A

N = 20
A = build_1d_upwind(N)

# Convert to dense for display
dense = np.zeros((N, N))
for i in range(N):
    row_cols, row_vals = A.getRow(i)
    for c, v in zip(row_cols, row_vals):
        dense[i, c] = v

fig, ax = plt.subplots(figsize=(5, 5))
ax.spy(dense, markersize=6)
ax.set_title(f'1D upwind matrix ({N}×{N})')
plt.tight_layout()
plt.show()

## 2. Compute PMISR-DDC CF splitting and define the DD metric

`compute_cf_splitting` is normally called on every level of the AIR hierarchy to define our multigrid levels, but we can call it independently to visualise the CF splitting on the top grid. It returns two PETSc `IS` (Index Set) objects: one for the F-points and one for the C-points.

Diagonal-dominance metric used throughout this notebook:
$$\text{dd}_i = \frac{\sum_{j \ne i} |a_{ij}|}{|a_{ii}|}.$$
Rows are diagonally dominant when $\text{dd}_i < 1$.

Parameter guide:
- `strong_threshold=0.5` → strength-of-connection cutoff which defines "strong" connections
- `ddc_its=1` → how many iterations of diagonal dominance correction we run after **PMISR**

In [ ]:
def run_cf_splitting(A, strong_threshold=0.5, ddc_its=1):
    is_fine, is_coarse = pflare.pflare_defs.compute_cf_splitting(
        A,
        False,
        strong_threshold,
        -1,
        pflare.CF_PMISR_DDC,
        ddc_its,
        0.1,
        0.0,
    )
    f_idx = set(is_fine.getIndices().tolist())
    c_idx = set(is_coarse.getIndices().tolist())
    is_fine.destroy()
    is_coarse.destroy()
    return f_idx, c_idx

def compute_aff_dd_ratios(A, f_idx):
    """Compute offdiag/diag ratio for each F-point row in A restricted to F×F."""
    f_list = sorted(f_idx)
    dd = []
    for i in f_list:
        cols, vals = A.getRow(i)
        diag = 0.0
        offdiag_sum = 0.0
        for c, v in zip(cols, vals):
            if c == i:
                diag = abs(v)
            elif c in f_idx:
                offdiag_sum += abs(v)
        if diag > 0:
            dd.append(offdiag_sum / diag)
        else:
            dd.append(float('inf'))
    return np.array(dd)

def print_aff_dd_summary(A, f_idx, c_idx=None, label='A_ff DD'):
    """Print worst-case DD ratio plus F/C sizes for the current splitting."""
    dd = compute_aff_dd_ratios(A, f_idx)
    n_f = len(f_idx)
    n_c = len(c_idx) if c_idx is not None else (A.getSize()[0] - n_f)

    if dd.size == 0:
        print(f'{label}: F={n_f}, C={n_c}, dd_ratio = n/a (no F-points)')
        return

    dd_ratio = float(np.max(dd))
    print(f'{label}: F={n_f}, C={n_c}, worst_row_dd_ratio = {dd_ratio:.6e}')
    return

f_idx, c_idx = run_cf_splitting(A)
print(f"coarsening ratio: {len(c_idx)/N:.2f}")
print(f"C-point indices: {sorted(c_idx)}")
print_aff_dd_summary(A, f_idx, c_idx, label='1D baseline')

## 3. Visualise C/F splitting

We can see that for a 1D stencil, the CF splitting has returned (close to an exact) red-black splitting. This gives a diagonal $A_{ff}$ (as there are no F-F connections) that can be easily approximated. In general we would not expect a red-black splitting on anything but the top level of a structured grid.

In [ ]:
def plot_cf_splitting(N, f_idx, c_idx, title='CF splitting'):
    colors = ['red' if i in f_idx else 'blue' for i in range(N)]
    fig, ax = plt.subplots(figsize=(10, 1.5))
    for i in range(N):
        ax.scatter(i, 0, c=colors[i], s=150, zorder=5)
    ax.set_xlim(-1, N)
    ax.set_yticks([])
    ax.set_xlabel('Node index')
    ax.set_title(title)
    red_p = mpatches.Patch(color='red', label='F-point')
    blue_p = mpatches.Patch(color='blue', label='C-point')
    ax.legend(handles=[red_p, blue_p], loc='upper right')
    plt.tight_layout()
    plt.show()

plot_cf_splitting(N, f_idx, c_idx,
    title=f'PMISR-DDC: threshold=0.5, ddc_its=1  ({len(c_idx)} C-points)')
print_aff_dd_summary(A, f_idx, c_idx, label='1D visualised case')

## 4. Build and visualise a 2D unstructured upwind example

To follow the 1D visualisation, we now use the existing Gmsh mesh `tests/data/square_unstruc.msh` and assemble a **node-centered upwind graph operator** on that unstructured triangulation.

Discretization used in this notebook section:
- A graph representation of an upwind operator with a finite-volume-like stencil
- For each node, only upwind neighbors contribute off-diagonal entries
- The diagonal is the sum of outgoing upwind weights, giving an asymmetric matrix suitable for PMISR-DDC
- N.b. This is not a FEM discretisation suitable for solving an advection operator! This is just for demonstrating the CF splitting.

In [ ]:
import matplotlib.tri as mtri
from pathlib import Path

def load_gmsh41_triangles(mesh_path):
    """Read node coordinates + triangle connectivity from a Gmsh v4.1 ASCII .msh file."""
    with open(mesh_path, 'r', encoding='utf-8') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    i0 = lines.index('$Nodes') + 1
    i1 = lines.index('$EndNodes')
    node_lines = lines[i0:i1]

    n_entity_blocks, _, _, _ = map(int, node_lines[0].split())
    cursor = 1
    node_coords = {}
    for _ in range(n_entity_blocks):
        entity_dim, entity_tag, parametric, n_in_block = map(int, node_lines[cursor].split())
        cursor += 1

        tags = [int(node_lines[cursor + k]) for k in range(n_in_block)]
        cursor += n_in_block

        for k in range(n_in_block):
            parts = node_lines[cursor + k].split()
            x, y = float(parts[0]), float(parts[1])
            node_coords[tags[k]] = (x, y)
        cursor += n_in_block

    j0 = lines.index('$Elements') + 1
    j1 = lines.index('$EndElements')
    elem_lines = lines[j0:j1]

    e_entity_blocks, _, _, _ = map(int, elem_lines[0].split())
    cursor = 1
    tri_tags = []
    for _ in range(e_entity_blocks):
        entity_dim, entity_tag, elem_type, n_in_block = map(int, elem_lines[cursor].split())
        cursor += 1

        for _ in range(n_in_block):
            vals = list(map(int, elem_lines[cursor].split()))
            cursor += 1
            if elem_type == 2 and entity_dim == 2:
                tri_tags.append(vals[1:4])

    used_tags = sorted({t for tri in tri_tags for t in tri})
    tag_to_idx = {tag: idx for idx, tag in enumerate(used_tags)}

    points = np.array([node_coords[tag] for tag in used_tags], dtype=float)
    triangles = np.array([[tag_to_idx[a], tag_to_idx[b], tag_to_idx[c]] for a, b, c in tri_tags], dtype=int)
    return points, triangles

def build_2d_unstructured_upwind_from_msh(mesh_path, velocity=(1.0, 0.6)):
    """Build a node-centered upwind graph operator on a triangular unstructured mesh."""
    pts, tris = load_gmsh41_triangles(mesh_path)
    tri = mtri.Triangulation(pts[:, 0], pts[:, 1], triangles=tris)
    n = pts.shape[0]

    neighbors = [set() for _ in range(n)]
    for t in tris:
        for a, b in ((t[0], t[1]), (t[1], t[2]), (t[2], t[0])):
            neighbors[a].add(b)
            neighbors[b].add(a)

    v = np.array(velocity, dtype=float)
    vnorm = np.linalg.norm(v)
    if vnorm == 0.0:
        raise ValueError('velocity must be non-zero')
    v /= vnorm

    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(12)
    A.setUp()

    rstart, rend = A.getOwnershipRange()
    eps = 1e-14
    for i in range(rstart, rend):
        xi = pts[i]
        upstream = []
        for j in neighbors[i]:
            d = xi - pts[j]
            dist2 = np.dot(d, d)
            if dist2 < eps:
                continue

            adv = np.dot(v, d)
            if adv > 0.0:
                w = adv / np.sqrt(dist2)
                upstream.append((j, w))

        row_sum = sum(w for _, w in upstream)
        if row_sum > 0.0:
            A.setValue(i, i, row_sum)
            for j, w in upstream:
                A.setValue(i, j, -w)
        else:
            A.setValue(i, i, 1.0)

    A.assemblyBegin()
    A.assemblyEnd()
    return A, pts, tri, v

mesh_rel = Path('tests/data/square_unstruc.msh')
search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
try:
    pflare_root = Path(pflare.__file__).resolve().parent.parent
    search_roots.extend([pflare_root, pflare_root.parent])
except Exception:
    pass

mesh_path = None
for root in search_roots:
    candidate = (root / mesh_rel).resolve()
    if candidate.exists():
        mesh_path = candidate
        break

if mesh_path is None:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / 'tests' / 'data' / 'square_unstruc.msh'
        if candidate.exists():
            mesh_path = candidate.resolve()
            break

if mesh_path is None:
    raise FileNotFoundError('Could not find tests/data/square_unstruc.msh; please run notebook from repo tree or set an absolute mesh path.')

A2, pts2, tri2, vel2 = build_2d_unstructured_upwind_from_msh(mesh_path, velocity=(1.0, 0.5))
n2 = A2.getSize()[0]

is_fine2, is_coarse2 = pflare.pflare_defs.compute_cf_splitting(
    A2, False, 0.5, -1, pflare.CF_PMISR_DDC, 1, 0.1, 0.0,
 )
f2 = set(is_fine2.getIndices().tolist())
c2 = set(is_coarse2.getIndices().tolist())
is_fine2.destroy()
is_coarse2.destroy()

dense2 = np.zeros((n2, n2))
for i in range(n2):
    cols, vals = A2.getRow(i)
    for c, val in zip(cols, vals):
        dense2[i, c] = val

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].triplot(tri2, color='0.55', lw=0.6)
axes[0].quiver(0.08, 0.08, vel2[0], vel2[1], angles='xy', scale_units='xy', scale=2.2, color='black')
axes[0].text(0.14, 0.09, 'flow direction', fontsize=9, va='center')
axes[0].set_title(f'Gmsh mesh: {mesh_path.name} ({n2} nodes)')
axes[0].set_aspect('equal')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

axes[1].spy(dense2, markersize=1.2)
axes[1].set_title('2D unstructured upwind matrix sparsity')
axes[1].set_xlabel('column')
axes[1].set_ylabel('row')

colors = ['red' if i in f2 else 'blue' for i in range(n2)]
axes[2].triplot(tri2, color='0.75', lw=0.5)
axes[2].scatter(pts2[:, 0], pts2[:, 1], c=colors, s=10)
axes[2].quiver(0.08, 0.08, vel2[0], vel2[1], angles='xy', scale_units='xy', scale=2.2, color='black')
axes[2].set_title(f'PMISR-DDC on unstructured 2D mesh ({len(c2)} C-points)')
axes[2].set_aspect('equal')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
red_p = mpatches.Patch(color='red', label='F-point')
blue_p = mpatches.Patch(color='blue', label='C-point')
axes[2].legend(handles=[red_p, blue_p], loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Loaded mesh: {mesh_path}')
print(f'A2 size: {n2} x {n2}, nnz={A2.getInfo()["nz_used"]:.0f}')
print_aff_dd_summary(A2, f2, c2, label='2D baseline')

Because this is an unstructured, directed upwind operator, PMISR-DDC no longer resembles a red-black pattern.

The coarsening occurs along the characteristics, so no F-F couplings occur along the velocity direction. The resulting $A_{ff}$ has no large off-diagonal entries.

## 5. Effect of `strong_threshold` on the 2D unstructured case

The `strong_threshold` changes the strength graph used by PMISR-DDC and therefore changes the C/F pattern and coarsening ratio. The previous example used `strong_threshold = 0.5`. We show two further cases here:

`strong_threshold = 0.0`:
- Every nonzero connection is treated as strong
- PMISR-DDC computes an exact independent set in the adjacency graph of the matrix
- Therefore F-points have no strong F-F connections, so $A_{ff}$ is diagonal
- This is a very slow coarsening and typically produces many multigrid levels

`strong_threshold = 0.99`:
- We would expect a very fast coarsening that still avoids the largest F-F connections in the flow direction

In [ ]:
thresholds = [0.0, 0.99]
fig, axes = plt.subplots(1, len(thresholds), figsize=(5 * len(thresholds), 4), sharex=True, sharey=True)
if len(thresholds) == 1:
    axes = [axes]

for ax, thresh in zip(axes, thresholds):
    f, c = run_cf_splitting(A2, strong_threshold=thresh)
    colors = ['red' if i in f else 'blue' for i in range(n2)]
    ax.triplot(tri2, color='0.80', lw=0.45)
    ax.scatter(pts2[:, 0], pts2[:, 1], c=colors, s=10)
    ax.quiver(0.08, 0.08, vel2[0], vel2[1], angles='xy', scale_units='xy', scale=2.2, color='black')
    ax.set_aspect('equal')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'strong_threshold={thresh}\nC={len(c)}, F={len(f)}')

    print_aff_dd_summary(A2, f, c, label=f'2D strong_threshold={thresh}')

red_p = mpatches.Patch(color='red', label='F-point')
blue_p = mpatches.Patch(color='blue', label='C-point')
fig.legend(handles=[red_p, blue_p], loc='upper right')
plt.tight_layout()
plt.show()

We can see that the `strong_threshold = 0.0` case has produced a very slow coarsening (very few F points), whereas the `strong_threshold = 0.99` case has coarsened very fast (more F points), but at the cost of a larger diagonal dominance ratio when compared to the `strong_threshold = 0.5` case.

## 6. Effect of `ddc_its` on diagonal dominance ratio

The **DDC** pass does a number of iterations, swapping the worst F points (measured by diagonal dominance ratio) to C points. The fraction of points it swaps per iteration is given by the `ddc_fraction` parameter, which we set to 0.1. Hence the worst 10% of F points are swapped to C points at each iteration. 

Using the same `ddc_fraction`, we increase the number of `ddc_its` and examine how it changes the $A_{ff}$ diagonal-dominance statistics.

In [ ]:
cases = [(0, 'ddc_its=0'), (1, 'ddc_its=1'), (3, 'ddc_its=3')]

for ddc_its, label in cases:
    f, c = run_cf_splitting(A2, ddc_its=ddc_its)
    print_aff_dd_summary(A2, f, c, label=f'2D {label}')

We can see that increasing the `ddc_its` improves the worst case diagonal dominace ratio, but slows down the coarsening, producing fewer F points. 

## 7. Diagonal dominance in the solve

We saw in the previous sections that we can produce a diagonally dominant submatrix $A_{ff}$ from a matrix $A$. We now use the GMRES polynomials from `PCPFLAREINV`, discussed in the previous notebook, to perform a solve with $A_{ff}$. Convergence criteria for GMRES (and GMRES polynomials) in non-normal systems is still an open question, but we observe that the more diagonally dominant, the faster convergence we see. 

Hence below we examine the solver performance as we pull out $A_{ff}$ matrices with different diagonal dominance ratios.

In [ ]:
def solve_aff_with_pflareinv(Aff, poly_order=6, sparsity_order=1, max_it=300, rtol=1e-10):
    prefix = f'affsolve__'
    opts = PETSc.Options()

    opt_keys = {
        'pc_pflareinv_type': 'arnoldi',
        'pc_pflareinv_poly_order': poly_order,
        'pc_pflareinv_sparsity_order': sparsity_order,
        'pc_pflareinv_matrix_free': 0,
    }
    for key, val in opt_keys.items():
        opts[prefix + key] = str(val)

    ksp = PETSc.KSP().create()
    ksp.setOptionsPrefix(prefix)
    ksp.setOperators(Aff)
    ksp.setType('richardson')
    ksp.setTolerances(rtol=rtol, max_it=max_it)
    ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
    ksp.getPC().setType('pflareinv')

    rnorms = []
    def monitor(ksp_obj, it, rnorm):
        rnorms.append(rnorm)
    ksp.setMonitor(monitor)

    ksp.setFromOptions()
    b = Aff.createVecRight(); b.set(1.0)
    x = Aff.createVecRight(); x.set(0.0)
    ksp.solve(b, x)

    for key in opt_keys.keys():
        try:
            del opts[prefix + key]
        except Exception:
            pass

    rnorms = np.array(rnorms, dtype=float)
    if rnorms.size > 0 and rnorms[0] != 0.0:
        rrel = rnorms / rnorms[0]
    else:
        rrel = rnorms
    return rrel, ksp.getIterationNumber()

thresholds_aff = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
curves_aff = []
stats_aff = []

for thresh in thresholds_aff:
    f_idx, c_idx = run_cf_splitting(A2, strong_threshold=thresh, ddc_its=1)
    f_sorted = np.array(sorted(f_idx), dtype=np.int32)
    if f_sorted.size == 0:
        print(f'strong_threshold={thresh}: no F points, skipping')
        continue

    is_f = PETSc.IS().createGeneral(f_sorted, comm=PETSc.COMM_WORLD)
    Aff = A2.createSubMatrix(is_f, is_f)

    dd_vals = compute_aff_dd_ratios(A2, f_idx)
    dd_max = float(np.max(dd_vals)) if dd_vals.size > 0 else float('inf')

    r_aff, it_aff = solve_aff_with_pflareinv(Aff, poly_order=6, sparsity_order=1, max_it=300, rtol=1e-10)
    x_aff = np.arange(1, len(r_aff) + 1, dtype=float)
    label = f'th={thresh}, dd_max={dd_max:.3f}, |F|={len(f_idx)}'
    curves_aff.append((label, x_aff, r_aff))
    stats_aff.append((thresh, dd_max, it_aff, len(f_idx)))

    print(f'strong_threshold={thresh}: |F|={len(f_idx)}, |C|={len(c_idx)}, dd_max={dd_max:.6f}, iters={it_aff}')

    Aff.destroy()
    is_f.destroy()

# Optional compact trend view: lower dd_max typically means fewer iterations
stats_aff_sorted = sorted(stats_aff, key=lambda t: t[1])
if len(stats_aff_sorted) > 1:
    dd_x = [s[1] for s in stats_aff_sorted]
    it_y = [s[2] for s in stats_aff_sorted]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(dd_x, it_y, '-o')
    ax.set_xlabel('worst-row dd ratio of A_ff (offdiag/diag)')
    ax.set_ylabel('Iterations to convergence')
    ax.set_title('More diagonally dominant A_ff -> better convergence')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

We can see that as $A_{ff}$ becomes less digaonally dominant (and the number of F points increases), convergence suffers. For all strong thresholds tested however, we can solve $A_{ff}$ to a relative tolerance of 1e-10 in only a few iterations. 

In comparison, if we try to solve the original $A$ matrix, we see it takes many more iterations, as it is much less diagonally dominant.

In [ ]:
# Solve original A with assembled PCPFLAREINV and report DD ratio + iteration count (unpreconditioned norm)
nA = A.getSize()[0]
all_idx = set(range(nA))

ksp_A = PETSc.KSP().create()
ksp_A.setOperators(A)
ksp_A.setType('richardson')
ksp_A.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
ksp_A.setTolerances(rtol=1e-10, max_it=500)

pc_A = ksp_A.getPC()
pc_A.setType('pflareinv')

opts_A = PETSc.Options()
prefix_A = f'origA_'
ksp_A.setOptionsPrefix(prefix_A)
opts_A[prefix_A + 'pc_pflareinv_type'] = 'arnoldi'
opts_A[prefix_A + 'pc_pflareinv_poly_order'] = '6'
opts_A[prefix_A + 'pc_pflareinv_sparsity_order'] = '1'
opts_A[prefix_A + 'pc_pflareinv_matrix_free'] = '0'

ksp_A.setFromOptions()
bA = A.createVecRight(); bA.set(1.0)
xA = A.createVecRight(); xA.set(0.0)
ksp_A.solve(bA, xA)

dd_A = float(np.max(compute_aff_dd_ratios(A, all_idx)))
iters_A = ksp_A.getIterationNumber()
print(f'original A: worst_row_dd_ratio={dd_A:.6e}, iterations={iters_A}')

for key in ['pc_pflareinv_type', 'pc_pflareinv_poly_order', 'pc_pflareinv_sparsity_order', 'pc_pflareinv_matrix_free']:
    try:
        del opts_A[prefix_A + key]
    except Exception:
        pass

## Summary

- Decreasing `strong_threshold` and increasing `ddc_its` (and/or `ddc_fraction`) slows down the coarsening and give a more diagonally dominant $A_{ff}$.
- Improving diagonal dominance of $A_{ff}$ improves the convergence of GMRES polynomials in the solve

## 8. What's next?

Now that we understand that we can pull out a diagonally dominant subset of unknowns (F-points) and we have seen how we can use the approximate inverses in `PCPFLAREINV` to solve those submatrices, we put those together and form an AIR multigrid method

**[04_pcair.ipynb](04_pcair.ipynb)**: Discuss PCAIR and AIRG